# YOLO Assignment – Subtask 4: Custom Object Detection on the Neural Ocean Dataset

**Task:** Train a custom YOLO object-detection model on Neural Ocean dataset version 3, then test it on multiple new images.

## Objective
Download Neural Ocean dataset v3 from Roboflow in YOLO format, fine-tune a pretrained YOLO nano model, inspect the training results and saved weights, and test the trained model on unseen images.

## Application
Detecting underwater debris and waste to support marine-environment monitoring.

## Dataset source and verified information
- Workspace: `neural-ocean`
- Project: `neural_ocean`
- Version: `3`
- Task: Object Detection
- Total images: **5,127**
- Train: **3,626 (71%)** | Validation: **1,000 (19%)** | Test: **501 (10%)**
- License shown on Roboflow Universe: **CC BY 4.0**
- Classes (15): `Mask`, `plastic`, `metal`, `can`, `glove`, `tire`, `sunglasses`, `cellphone`, `electronics`, `gbottle`, `misc`, `net`, `pbag`, `pbottle`, `rod`.

**Dataset:** https://universe.roboflow.com/neural-ocean/neural_ocean/dataset/3


##  Prepare the Environment

**Objective:** Install the libraries needed to download the dataset, train YOLO, inspect images, and display predictions.

**Why:** Ultralytics trains YOLO; Roboflow downloads the dataset; OpenCV and Matplotlib display images; PyYAML reads `data.yaml`.

In [1]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="JMy8627CEJtSvlK5pED1")
project = rf.workspace("neural-ocean").project("neural_ocean")
version = project.version(3)
dataset = version.download("yolo26")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 60.1 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [roboflow]
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Neural_Ocean-3 in yolo26:: 100%|██████████| 10266/10266 [00:01<00:00, 8830.79it/s]


In [7]:
pip install ultralytics

In [3]:
pip install opencv-python

### import libraries

In [4]:
import os, yaml
import pandas as pd
import numpy as np
import cv2
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from ultralytics import YOLO
from roboflow import Roboflow

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


##  Train the Custom YOLO Model

**Objective:** Train YOLO11n on Neural Ocean.

| Parameter | Value | Purpose |
|---|---:|---|
| Task | Detection | Find objects and draw bounding boxes |
| Model | YOLO11n | Nano model |
| Epochs | 60 | Up to 60 full passes through training data |
| Image size | 640 | More detail, but higher memory use |
| Optimizer | SGD | Updates model weights |
| Learning rate | 0.001 | Low starting update size |

**Why parameters matter:** Epochs control training duration; image size affects detail and memory; optimizer and learning rate affect weight updates; patience can stop training when validation performance plateaus.


##  Inspect Training Results and Weights

**Objective:** Locate `best.pt`, `last.pt`, metrics, and plots.

- `best.pt`: weights from the epoch with the best validation fitness.
- `last.pt`: weights from the last completed epoch.
- `results.csv`: numerical training metrics.
- `results.png`: training and validation curves.
- Confusion matrix and precision/recall plots may also be generated.

**Why inspect them?** They help us understand training quality and choose the checkpoint for final prediction.

In [5]:
from ultralytics import YOLO

# Load a model
model = YOLO("yolo26n.pt")  # load a pretrained model (recommended for training)

# Train the model
results = model.train(data="/content/Neural_Ocean-3/data.yaml", epochs=60, imgsz=640, # optimizer="SGD",
#lr0=0.001
                      )

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/Neural_Ocean-3/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, op

In [6]:

# Load a model
model = YOLO("/content/runs/detect/train/weights/best.pt")  # load a custom model

# Validate the model
metrics = model.val()  # no arguments needed, dataset and settings remembered
metrics.box.map  # map50-95
metrics.box.map50  # map50
metrics.box.map75  # map75
metrics.box.maps  # mAP50-95 per class; classes absent from the val set hold the overall mAP
metrics.box.ap_class_index  # class indices that were actually evaluated
metrics.box.image_metrics  # per-image metrics dictionary with precision, recall, F1, TP, FP, and FN

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n summary (fused): 121 layers, 2,393,137 parameters, 0 gradients, 5.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1027.3±557.4 MB/s, size: 22.5 KB)
val: Scanning /content/Neural_Ocean-3/valid/labels.cache... 1000 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1000/1000 279.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 63/63 6.4it/s 9.8s
                   all       1000       1891      0.834      0.749      0.813      0.517
                  Mask         77         90      0.928      0.576      0.836       0.63
                   can         18         20      0.848      0.835      0.812      0.338
             cellphone         61         71       0.99      0.972       0.99      0.863
           electronics         27         40       0.87        0.8      0.838      0.472
               gbottle         36 

{'007104744_jpg.rf.1a6049ee02ec45b70d6a3414d99befc9.jpg': {'precision': 0.08333333333333333,
  'recall': 1.0,
  'f1': 0.15384615384615385,
  'tp': 1,
  'fp': 11,
  'fn': 0},
 '007105123_jpg.rf.f6b4e0c59b9e1c15c9e385011f371c8c.jpg': {'precision': 0.1,
  'recall': 1.0,
  'f1': 0.18181818181818182,
  'tp': 1,
  'fp': 9,
  'fn': 0},
 '0211007104833_jpg.rf.e950f18d0b8c32bc6954ccabdc3cd8da.jpg': {'precision': 0.125,
  'recall': 1.0,
  'f1': 0.2222222222222222,
  'tp': 1,
  'fp': 7,
  'fn': 0},
 '0211007104855_jpg.rf.9f911e0d2611e3038f446e0943f4f373.jpg': {'precision': 0.14285714285714285,
  'recall': 1.0,
  'f1': 0.25,
  'tp': 1,
  'fp': 6,
  'fn': 0},
 '0211007104914_jpg.rf.4f3b8efa52ab6ba6919d2d712fb6ed05.jpg': {'precision': 0.2,
  'recall': 1.0,
  'f1': 0.33333333333333337,
  'tp': 1,
  'fp': 4,
  'fn': 0},
 '0211007105142_jpg.rf.57024f3ac7b02345ff86c75ee1c156f2.jpg': {'precision': 0.14285714285714285,
  'recall': 1.0,
  'f1': 0.25,
  'tp': 1,
  'fp': 6,
  'fn': 0},
 '02110071052097_jpg.r

##  Evaluate on the Held-Out Test Split

**Objective:** Evaluate `best.pt` on the dataset test split, if the exported YAML includes a test path.

**Why?** The test split contains held-out images and gives an additional check of generalization.

In [17]:


# Predict with the model
results = model("/content/photo_8_2026-11-17_14-55-50.jpg",save = True)  # predict on an image

# Access the results
for result in results:
    xywh = result.boxes.xywh  # center-x, center-y, width, height
    xywhn = result.boxes.xywhn  # normalized
    xyxy = result.boxes.xyxy  # top-left-x, top-left-y, bottom-right-x, bottom-right-y
    xyxyn = result.boxes.xyxyn  # normalized
    names = [result.names[cls.item()] for cls in result.boxes.cls.int()]  # class name of each box
    confs = result.boxes.conf  # confidence score of each box


image 1/1 /content/photo_8_2026-11-17_14-55-50.jpg: 384x640 2 gloves, 14.7ms
Speed: 1.9ms preprocess, 14.7ms inference, 1.6ms postprocess per image at shape (1, 3, 384, 640)
Results saved to /content/runs/detect/predict


### Export model

In [18]:
from ultralytics import YOLO

# Load a model
model = YOLO("/content/runs/detect/train/weights/best.pt")  # load a custom model

# Export the model
model.export(format="onnx")

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
YOLO26n summary (fused): 121 layers, 2,393,137 parameters, 0 gradients, 5.4 GFLOPs

PyTorch: starting from '/content/runs/detect/train/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 19, 8400) (5.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 283ms
Prepared 4 packages in 685ms
Installed 4 packages in 39ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.32.0
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 1.3s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming wi

'/content/runs/detect/train/weights/best.onnx'

Required Questions and Answers

### 1. What is the dataset used for?
The Neural Ocean dataset is an object-detection dataset intended for detecting underwater debris and garbage.

### 2. What classes exist in the dataset?
The project page lists 15 classes: `Mask`, `plastic`, `metal`, `can`, `glove`, `tire`, `sunglasses`, `cellphone`, `electronics`, `gbottle`, `misc`, `net`, `pbag`, `pbottle`, and `rod`. Verify the exact IDs and names in the downloaded `data.yaml`.

### 3. How is the dataset organized after download?
A YOLO detection dataset contains images and matching label text files, usually separated into training, validation, and test splits. `data.yaml` describes the paths and class names.

### 4. Why do training parameters matter?
They affect learning speed, memory use, convergence, and generalization.

### 5. What does image size affect?
Image size affects the detail available to the model, computation, and memory use. Larger images may help small objects but require more resources.

### 6. Why is patience used?
Patience enables early stopping if validation performance does not improve for the configured number of epochs.

### 7. How do optimizer and learning rate affect training?
The optimizer determines how model weights are updated. The learning rate controls the size of each update.

### 8. Which file should be used for final prediction?
Usually `best.pt`, because it is saved from the epoch with the best validation fitness.

### 9. Why is saving model weights important?
Saved weights preserve what the model learned and can be loaded later without repeating training.

### 10. What does confidence threshold mean?
It is the minimum confidence score required for a detection to be kept.

### 11. Why may some images fail detection?
An object may be small, blurred, hidden, poorly lit, photographed from an unfamiliar angle, or outside the dataset's classes.

### 12. Why is testing on unseen images important?
It checks whether the model generalizes to images it did not learn from, rather than only memorizing training examples.